# 07 - Parking-slot geometry and occupancy (YOLO11n and YOLO11s, fixed model, no controller yet)
Turns vehicle detections into **slot occupancy** and measures it against the MetaPKLot slot labels.

1. **Slot registry**: one persistent `slot_id` per camera and polygon (polygons are fixed per camera), with reference size, coordinate system and provenance. Validated: points in range, nonzero area, no self-intersection, unique ids, no slot twice in one image, and the exported images have the reference size (so no coordinate transform is needed).
2. **Detections**: each model runs once on the val and test images (the ROI-masked export from notebook 03) at a low confidence floor. Detections are cached, so later notebooks (temporal, controller, cost) reuse them.
3. **Association**, three methods from the spec: box/slot IoU, intersection / slot area, box centre inside the slot. Each with and without *exclusive* assignment (one detection fills at most one slot). The method, overlap threshold and score threshold are **chosen on val only** (highest macro-F1 over occupied/vacant).
4. **Test** at the val-chosen setting: accuracy, occupied/vacant precision/recall/F1, false-vacant and false-occupied counts, free-slot count error per image, per site, camera, climate and day type, and where YOLO11n is wrong but YOLO11s is right.
5. Per-slot output records in the spec schema, slot scores for val and test (input to notebooks 08-11), plots and overlays.

The slot score is the **uncalibrated** detector confidence of the best supporting detection, so the threshold is a *score threshold*, not a probability (calibration and cost ratios come in notebook 11). A slot mostly outside the annotation rectangle (gray in the export) is reported **unknown**, never vacant.

**Inputs (Add Input -> Your Work):** notebook 03 output (`yolo_f0.1/`, `artifacts/splits/`), notebook 04 output (yolo11n), notebook 05 output (yolo11s), and either **notebook 02 output** (`annotations_v2/`, faster) or the **MetaPKLot dataset** (slot polygons and labels).
**Accelerator:** GPU T4 x2 (one GPU is used). Internet ON (installs `ultralytics==8.3.40`).

**How to run:** `SMOKE = True` + Run All (~5 min, 150 val and 150 test images). Then `SMOKE = False` + **Save Version -> Save & Run All**. Re-running reuses cached detections and records with the same config hash.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Install the pinned ultralytics (same version as the Orin so the .pt loads there). No torch upgrade.
import importlib.metadata as md_
try:
    have = md_.version("ultralytics")
except md_.PackageNotFoundError:
    have = None
if have != "8.3.40":
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.3.40"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    print(r.stdout[-2000:])
# torch >= 2.6 defaults torch.load(weights_only=True), which ultralytics 8.3.40 checkpoints do not satisfy.
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")
# Training uses one GPU. Hiding the second one lets ultralytics use all 4 CPU cores for data loading
# (it divides the cores by the number of visible GPUs: 2 loader workers on "T4 x2"). Must be set before torch loads.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import torch, ultralytics
# Turn off experiment-tracker integrations (Kaggle's ray breaks the raytune callback; wandb can wait for a login).
# Runs before any trainer exists. If you saw a raytune error earlier, restart the kernel first.
from src.training import yolo_train as yt
print("tracker integrations:", yt.disable_integrations())
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# Settings and inputs
SMOKE = True                      # True: 150 val + 150 test images. False: full val and test splits (use Save & Run All)
VARIANTS = ["yolo11n", "yolo11s"]
DEVICE = 0
BATCH = 32
SMOKE_IMAGES = 150
PRED_CONF = 0.05                  # confidence floor of the cached detections (the score threshold is chosen above it on val)
NMS_IOU = 0.7                     # ultralytics default
MIN_ROI = 0.5                     # a slot with less than this fraction inside the annotation rectangle is 'unknown'
DATASET_HINT = ""                 # only if several datasets are attached and no notebook 02 output is attached
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import json, re, time, csv, gzip
from pathlib import Path
import numpy as np
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.common.hashing import sha256_file
from src.training import yolo_train as yt
from src.evaluation import det_eval as de
from src.data import metapklot as mp
from src.data.yolo_export import safe_name
from src.occupancy import geometry as geo, association as asc, schema
DCFG = load_yaml("configs/dataset.yaml")
OCFG = load_yaml("configs/occupancy.yaml")
FRAC = DCFG["dataset"]["subset_fraction"]
IMGSZ = {v: load_yaml("configs/train_{}.yaml".format(v))["imgsz"] for v in VARIANTS}
EXPORT = yt.find_export_dir(INPUT_ROOT, FRAC)
mans = [p for p in yt.find_files(INPUT_ROOT, "split_manifest.csv") if p.parent.name == "splits"]
assert mans, "split_manifest.csv not found: attach notebook 03's output"
ROWS = de.read_manifest(mans[0])
SPLIT_INFO = json.loads((mans[0].parent / "split_info.json").read_text())
print("export:", EXPORT, "| manifest:", mans[0], "| hash:", SPLIT_INFO["manifest_hash"])

MODELS, MISSING = {}, []
records = yt.find_files(INPUT_ROOT, "training_record.json")
for v in VARIANTS:
    name = "{}_f{}".format(v, FRAC)
    found = [p for p in records if p.parent.name == name]
    if not found:
        MISSING.append(v); print("MISSING", name, "- attach the output of the notebook that trained it"); continue
    d = found[0].parent
    rec = json.loads(found[0].read_text())
    sha = sha256_file(d / "best.pt")
    assert sha == rec["checkpoint_sha256"], "{}: best.pt does not match its training record".format(v)
    assert not rec.get("smoke_test"), "{}: this is a smoke-test checkpoint".format(v)
    assert rec["dataset_split_hash"] == SPLIT_INFO["manifest_hash"], "{}: trained on a different split".format(v)
    MODELS[v] = {"weights": d / "best.pt", "sha256": sha}
    print("{}: {} | sha256 {}...".format(v, d, sha[:12]))
assert MODELS, "no trained model attached"

# Slot polygons and labels: notebook 02 output if attached, else the dataset itself
t0 = time.time()
nb02 = [p for p in yt.find_files(INPUT_ROOT, "spots.csv.gz") if p.parent.name == "annotations_v2"]
if nb02:
    base = nb02[0].parent
    IMAGES = {}
    with open(str(base / "images.csv")) as f:
        for r in csv.DictReader(f):
            IMAGES[int(r["image_id"])] = {"image_id": int(r["image_id"]), "file_name": r["file_name"], "camera": r["camera"],
                                          "width": int(r["width"]), "height": int(r["height"]),
                                          "rect": json.loads(r["rect"]) if r["rect"] else None}
    SPOTS = []
    with gzip.open(str(base / "spots.csv.gz"), "rt") as f:
        for r in csv.DictReader(f):
            SPOTS.append({"ann_id": int(r["ann_id"]), "image_id": int(r["image_id"]), "category_id": int(r["category_id"]),
                          "car_id": int(r["car_id"]), "polygon": json.loads(r["polygon"])})
    ANN_SOURCE = "notebook 02 output " + str(base)
else:
    from src.data.discover import find_dataset_root
    root = find_dataset_root(INPUT_ROOT, DATASET_HINT)
    S = mp.load_set(root, DCFG["dataset"]["annotation_variant"], "spots", DCFG["dataset"]["sources"])
    IMAGES = {i["image_id"]: i for i in S["images"]}
    SPOTS = S["annotations"]
    ANN_SOURCE = "dataset " + str(root)
PROVENANCE = "MetaPKLot annotations/{}/spots (via {})".format(DCFG["dataset"]["annotation_variant"], ANN_SOURCE)
print("annotations:", ANN_SOURCE, "| images:", len(IMAGES), "| spots:", len(SPOTS), "| loaded in", round(time.time() - t0, 1), "s")

# Copy val+test images to local disk (the input mount is slow)
LOCAL = Path(WORK) / "yolo_local_f{}".format(FRAC)
t0 = time.time(); print("staging:", yt.stage_export(EXPORT, LOCAL, splits=("val", "test")), "in", round(time.time() - t0, 1), "s")
OUT = Path(WORK) / "artifacts" / "occupancy"
OUT.mkdir(parents=True, exist_ok=True)
SUF = "_smoke" if SMOKE else ""

In [ ]:
# Slot registry and validation (geometry only: uses every annotated image, no labels are tuned here)
from collections import defaultdict
t0 = time.time()
SLOTS, SID = geo.build_registry(IMAGES.values(), SPOTS, PROVENANCE)
VALID = geo.validate_slots(SLOTS, {s["ann_id"]: (s["image_id"], SID[s["ann_id"]]) for s in SPOTS if s["ann_id"] in SID})
# coordinate transform check: exported images must have the reference size the polygons were drawn on
from PIL import Image
by_name = {safe_name(r["file_name"]): int(r["image_id"]) for r in ROWS}
NAMES = {s: sorted(p.name for p in (LOCAL / "images" / s).iterdir()) for s in ("val", "test")}
size_bad = []
for s in ("val", "test"):
    for n in de.stride_sample(NAMES[s], 20):
        im = IMAGES[by_name[n]]
        with Image.open(str(LOCAL / "images" / s / n)) as pim:
            if pim.size != (im["width"], im["height"]):
                size_bad.append((n, pim.size, (im["width"], im["height"])))
VALID["transform"] = {"checked": 40, "size_mismatch": len(size_bad), "sample": size_bad[:3],
                      "rule": "identity (export keeps the original resolution)" if not size_bad else "MISMATCH"}
print("slots:", VALID["slots"], "| ok:", VALID["ok"], "| transform:", VALID["transform"]["rule"], "| built in", round(time.time() - t0, 1), "s")
for k in ("duplicate_slot_ids", "same_slot_twice_in_one_image", "out_of_range", "zero_area", "not_simple", "too_few_points", "mixed_ref_sizes"):
    print("  {:<30} {}".format(k, VALID[k]))
print("  slots per camera:", VALID["slots_per_camera"])
assert not size_bad, "exported image size differs from the annotation size: coordinates would need a transform"
(OUT / "slots.json").write_text(json.dumps(list(SLOTS.values()), indent=1))
(OUT / "slot_validation.json").write_text(json.dumps(VALID, indent=2, default=str))

# Slot observations of val and test: one row per (image, slot)
spots_by_img = defaultdict(list)
for s in SPOTS:
    spots_by_img[s["image_id"]].append(s)
if SMOKE:
    NAMES = {s: de.stride_sample(n, SMOKE_IMAGES) for s, n in NAMES.items()}
ROW_OF = {safe_name(r["file_name"]): r for r in ROWS}
roi_cache = {}
OBS = {}
for split in ("val", "test"):
    cols = defaultdict(list); starts = []
    for k, n in enumerate(NAMES[split]):
        iid = by_name[n]; im = IMAGES[iid]
        starts.append(len(cols["slot_id"]))
        for s in sorted(spots_by_img.get(iid, []), key=lambda s: SID[s["ann_id"]]):
            sid = SID[s["ann_id"]]
            key = (sid, tuple(im["rect"]) if im["rect"] else None)
            if key not in roi_cache:
                roi_cache[key] = geo.roi_fraction(SLOTS[sid]["polygon"], im["rect"])
            cols["image"].append(k); cols["slot_id"].append(sid); cols["gt"].append(int(s["category_id"] == 1))
            cols["roi"].append(roi_cache[key])
    o = {c: np.array(v) for c, v in cols.items()}
    o["valid"] = o["roi"] >= MIN_ROI
    o["starts"] = np.array(starts + [len(o["slot_id"])])
    OBS[split] = o
    print("{:<5} images {:>5} | slot observations {:>7} | occupied {:.1%} | unknown (ROI < {:.0%}) {}".format(
        split, len(NAMES[split]), len(o["gt"]), o["gt"][o["valid"]].mean(), MIN_ROI, int((~o["valid"]).sum())))

In [ ]:
# Detections on val and test, cached (restartable; later notebooks reuse them)
from ultralytics import YOLO
DET_DIR = OUT / "detections"
DET_DIR.mkdir(exist_ok=True)
CFG_HASH = config_hash({"dataset": DCFG, "occupancy": OCFG, "smoke": SMOKE, "smoke_images": SMOKE_IMAGES, "imgsz": IMGSZ,
                        "pred_conf": PRED_CONF, "nms_iou": NMS_IOU, "min_roi": MIN_ROI, "split": SPLIT_INFO["manifest_hash"],
                        "weights": {v: m["sha256"] for v, m in MODELS.items()}})
RUN_ID = new_run_id("occ")
STATUS = OUT / "status.json"
write_status(STATUS, "07_slot_occupancy", RUN_ID, "RUNNING", CFG_HASH, "detections")
DETS, DET_META = {}, {}
try:
    for v, m in MODELS.items():
        model = None
        for split in ("val", "test"):
            f = DET_DIR / "{}_{}{}.npz".format(v, split, SUF)
            meta_f = f.with_suffix(".json")
            if f.exists() and meta_f.exists() and json.loads(meta_f.read_text()).get("config_hash") == CFG_HASH:
                z = np.load(str(f)); DETS[(v, split)] = {"dets": z["dets"], "starts": z["starts"]}
                DET_META[(v, split)] = json.loads(meta_f.read_text()); print(v, split, ": cached"); continue
            model = model or YOLO(str(m["weights"]))
            paths = [str(LOCAL / "images" / split / n) for n in NAMES[split]]
            rows_, starts, speed, t0 = [], [0], [], time.time()
            for r in model.predict(paths, conf=PRED_CONF, iou=NMS_IOU, imgsz=IMGSZ[v], device=DEVICE, half=True,
                                   batch=BATCH, max_det=300, verbose=False, stream=True):
                b = r.boxes
                arr = np.concatenate([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()[:, None]], 1) if len(b) else np.zeros((0, 5))
                rows_.append(arr.astype(np.float32)); starts.append(starts[-1] + len(arr)); speed.append(r.speed["inference"])
            dets = np.concatenate(rows_) if rows_ else np.zeros((0, 5), np.float32)
            assert len(starts) == len(paths) + 1, "prediction count does not match the image list"
            np.savez_compressed(str(f), dets=dets, starts=np.array(starts), names=np.array(NAMES[split]))
            meta = {"config_hash": CFG_HASH, "model": v, "checkpoint_sha256": m["sha256"], "split": split, "images": len(paths),
                    "detections": int(len(dets)), "conf_floor": PRED_CONF, "nms_iou": NMS_IOU, "imgsz": IMGSZ[v], "half": True,
                    "columns": ["x0", "y0", "x1", "y1", "conf"], "coords": geo.COORD_SYSTEM,
                    "inference_ms_per_image_kaggle_t4_streamed_p50": round(float(np.median(speed)), 3) if speed else None,
                    "seconds": round(time.time() - t0, 1)}
            meta_f.write_text(json.dumps(meta, indent=1))
            DETS[(v, split)] = {"dets": dets, "starts": np.array(starts)}; DET_META[(v, split)] = meta
            print("{} {}: {} images, {} detections (conf >= {}), {:.0f} s".format(v, split, len(paths), len(dets), PRED_CONF, meta["seconds"]))
except Exception as e:
    write_status(STATUS, "07_slot_occupancy", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500])
    raise

In [ ]:
# Association pairs, then choose method + thresholds on VAL only
def build_pairs(v, split):
    o, D = OBS[split], DETS[(v, split)]
    parts, prep = [], {}
    for k in range(len(NAMES[split])):
        a, b = o["starts"][k], o["starts"][k + 1]
        if a == b:
            continue
        ids = tuple(o["slot_id"][a:b])
        if ids not in prep:
            prep[ids] = asc.prepare_slots([SLOTS[s]["polygon"] for s in ids])
        d0, d1 = D["starts"][k], D["starts"][k + 1]
        p = asc.image_pairs(prep[ids], D["dets"][d0:d1])
        p["slot"] += a; p["det"] += d0
        parts.append(p)
    return np.concatenate(parts) if parts else asc.image_pairs(asc.prepare_slots([]), np.zeros((0, 5)))

PAIRS, SWEEP, CHOSEN = {}, {}, {}
try:
    for v in MODELS:
        for split in ("val", "test"):
            t0 = time.time(); PAIRS[(v, split)] = build_pairs(v, split)
            print("{} {}: {} slot-detection pairs in {:.0f} s".format(v, split, len(PAIRS[(v, split)]), time.time() - t0))
        o = OBS["val"]
        rows_ = asc.sweep(len(o["gt"]), PAIRS[(v, "val")], o["gt"], o["valid"], asc.conf_grid(PRED_CONF))
        SWEEP[v] = rows_
        with open(str(OUT / "val_sweep_{}{}.csv".format(v, SUF)), "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows_[0])); w.writeheader(); w.writerows(rows_)
        CHOSEN[v] = asc.best(rows_)
        print("\n{} - best per method on VAL (macro-F1 over occupied/vacant):".format(v))
        print("  {:<24} {:<9} {:>7} {:>6} {:>9} {:>8} {:>7} {:>7}".format("method", "exclusive", "overlap", "score", "macro_F1", "accuracy", "FV_rate", "FO_rate"))
        for meth in asc.METHODS:
            b = asc.best(rows_, method=meth)
            print("  {:<24} {:<9} {:>7} {:>6} {:>9} {:>8} {:>7} {:>7}{}".format(meth, str(b["exclusive"]), str(b["overlap_thr"]), b["conf_thr"],
                  b["macro_f1"], b["accuracy"], b["false_vacant_rate"], b["false_occupied_rate"], "   <- chosen" if b is CHOSEN[v] else ""))
except Exception as e:
    write_status(STATUS, "07_slot_occupancy", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500])
    raise

In [ ]:
# TEST at the val-chosen setting: overall, per group, free-slot count error, YOLO11n vs YOLO11s
def scores_for(v, split):
    c = CHOSEN[v]
    return asc.slot_scores(len(OBS[split]["gt"]), PAIRS[(v, split)], c["method"], c["overlap_thr"], c["exclusive"])

def count_error(o, pred):
    # per image: |predicted free slots - true free slots| over the valid slots
    err = []
    for k in range(len(o["starts"]) - 1):
        a, b = o["starts"][k], o["starts"][k + 1]
        m = o["valid"][a:b]
        if m.any():
            err.append(abs(int((~pred[a:b][m]).sum()) - int((o["gt"][a:b][m] == 0).sum())))
    e = np.array(err) if err else np.zeros(1)
    return {"images": len(err), "mae": round(float(e.mean()), 3), "p90": float(np.percentile(e, 90)), "max": int(e.max()),
            "exact_share": round(float((e == 0).mean()), 4)}

o = OBS["test"]
img_rows = [ROW_OF[n] for n in NAMES["test"]]
obs_groups = defaultdict(list)          # group -> observation indices
for k, r in enumerate(img_rows):
    idx = range(o["starts"][k], o["starts"][k + 1])
    for g in ["all"] + de.group_keys(r):
        obs_groups[g].extend(idx)
obs_groups = {g: np.array(v, dtype=int) for g, v in sorted(obs_groups.items())}
img_count = defaultdict(set); day_count = defaultdict(set)
for r in img_rows:
    for g in ["all"] + de.group_keys(r):
        img_count[g].add(r["file_name"]); day_count[g].add((r["site"], r["day"]))

SCORES, PRED, RESULTS = {}, {}, {}
for v in MODELS:
    c = CHOSEN[v]
    for split in ("val", "test"):
        SCORES[(v, split)] = scores_for(v, split)
        PRED[(v, split)] = SCORES[(v, split)] >= c["conf_thr"]
    p, gt, valid = PRED[(v, "test")], o["gt"].astype(bool), o["valid"]
    groups = {}
    for g, idx in obs_groups.items():
        idx = idx[valid[idx]]
        s = asc.confusion(gt[idx], p[idx])
        s.update(images=len(img_count[g]), days=len(day_count[g]), note=de.small_group_note(len(img_count[g]), len(day_count[g])))
        groups[g] = s
    RESULTS[v] = {"run_id": RUN_ID, "config_hash": CFG_HASH, "smoke_test": SMOKE, "model_variant": v,
                  "checkpoint_sha256": MODELS[v]["sha256"], "dataset_split_hash": SPLIT_INFO["manifest_hash"],
                  "chosen_on_val": {k: c[k] for k in ("method", "exclusive", "overlap_thr", "conf_thr", "macro_f1", "accuracy",
                                                       "false_vacant_rate", "false_occupied_rate")},
                  "selection_rule": "max macro-F1 (occupied/vacant) on val; ties: accuracy, non-exclusive, lower thresholds",
                  "best_per_method_on_val": {m_: {k: asc.best(SWEEP[v], method=m_)[k] for k in ("exclusive", "overlap_thr", "conf_thr", "macro_f1", "accuracy")} for m_ in asc.METHODS},
                  "test": groups["all"], "test_free_slot_count_error": count_error(o, p),
                  "test_unknown_slots": int((~valid).sum()), "groups": {g: s for g, s in groups.items() if g != "all"},
                  "score_type": "uncalibrated detector score (score threshold, not a probability)",
                  "detections": {s_: DET_META[(v, s_)] for s_ in ("val", "test")}}
    (OUT / "occupancy_record_{}{}.json".format(v, SUF)).write_text(json.dumps(RESULTS[v], indent=2, default=str))

# where does the bigger model fix the small one? (test, valid slots, each model at its own val-chosen setting)
if len(MODELS) == 2:
    n_ok = PRED[("yolo11n", "test")] == o["gt"].astype(bool); s_ok = PRED[("yolo11s", "test")] == o["gt"].astype(bool)
    NS = {}
    for g, idx in obs_groups.items():
        idx = idx[o["valid"][idx]]
        NS[g] = {"slots": len(idx), "both_right": int((n_ok & s_ok)[idx].sum()), "n_wrong_s_right": int((~n_ok & s_ok)[idx].sum()),
                 "n_right_s_wrong": int((n_ok & ~s_ok)[idx].sum()), "both_wrong": int((~n_ok & ~s_ok)[idx].sum())}
    (OUT / "n_vs_s_test{}.json".format(SUF)).write_text(json.dumps(NS, indent=2))

# tables
print("TEST SPLIT (slot level, each model at its val-chosen setting; unknown slots excluded)")
for v, r in RESULTS.items():
    t = r["test"]
    print("{}: {} | slots {} | acc {} | macro-F1 {} | occupied P/R {}/{} | FV {} ({:.2%}) | FO {} ({:.2%}) | free-count MAE {} per image | unknown {}".format(
        v, {k: r["chosen_on_val"][k] for k in ("method", "exclusive", "overlap_thr", "conf_thr")}, t["slots"], t["accuracy"], t["macro_f1"],
        t["occupied_precision"], t["occupied_recall"], t["FV"], t["false_vacant_rate"], t["FO"], t["false_occupied_rate"],
        r["test_free_slot_count_error"]["mae"], r["test_unknown_slots"]))
csv_rows = []
for v, r in RESULTS.items():
    for g, s in [("all", r["test"])] + sorted(r["groups"].items()):
        csv_rows.append({"model": v, "group": g, "images": s["images"], "days": s["days"], "slots": s["slots"], "accuracy": s["accuracy"],
                         "macro_f1": s["macro_f1"], "FV": s["FV"], "FO": s["FO"], "false_vacant_rate": s["false_vacant_rate"],
                         "false_occupied_rate": s["false_occupied_rate"], "note": s["note"],
                         **({"n_wrong_s_right": NS[g]["n_wrong_s_right"], "n_right_s_wrong": NS[g]["n_right_s_wrong"]} if len(MODELS) == 2 else {})})
with open(str(OUT / "groups_test{}.csv".format(SUF)), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(csv_rows[0])); w.writeheader(); w.writerows(csv_rows)
print()
print("{:<8} {:<30} {:>6} {:>7} {:>7} {:>8} {:>6} {:>6} {:>7} {:>7}  {}".format("model", "group", "images", "slots", "acc", "macroF1", "FV", "FO", "n-s+", "n+s-", "note"))
for r_ in csv_rows:
    print("{:<8} {:<30} {:>6} {:>7} {:>7} {:>8} {:>6} {:>6} {:>7} {:>7}  {}".format(*[str(r_.get(k, "")) for k in (
          "model", "group", "images", "slots", "accuracy", "macro_f1", "FV", "FO", "n_wrong_s_right", "n_right_s_wrong", "note")]))

# slot scores for later notebooks (08 temporal, 09-10 controller, 11 cost): one row per (image, slot)
for split in ("val", "test"):
    oo = OBS[split]
    with gzip.open(str(OUT / "slot_scores_{}{}.csv.gz".format(split, SUF)), "wt", newline="") as f:
        w = csv.writer(f)
        w.writerow(["image", "image_id", "camera", "timestamp", "slot_id", "gt_occupied", "roi_fraction", "valid"] + ["score_" + v for v in MODELS])
        for k, n in enumerate(NAMES[split]):
            r = ROW_OF[n]
            for i in range(oo["starts"][k], oo["starts"][k + 1]):
                w.writerow([n, r["image_id"], r["camera"], r["timestamp"], oo["slot_id"][i], int(oo["gt"][i]), round(float(oo["roi"][i]), 4),
                            int(oo["valid"][i])] + [round(float(SCORES[(v, split)][i]), 4) for v in MODELS])
print("\nsaved slot scores for val and test, and", [p.name for p in sorted(OUT.glob("*" + SUF + ".*"))])

In [ ]:
# Plots: val macro-F1 vs score threshold per method, test accuracy / false-vacant per group, and slot overlays
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(MODELS), figsize=(6.5 * len(MODELS), 4.2), squeeze=False)
for a, v in zip(ax[0], MODELS):
    for meth in asc.METHODS:
        b = asc.best(SWEEP[v], method=meth)
        pts = sorted((r["conf_thr"], r["macro_f1"]) for r in SWEEP[v]
                     if r["method"] == meth and r["exclusive"] == b["exclusive"] and r["overlap_thr"] == b["overlap_thr"])
        a.plot(*zip(*pts), label="{} (excl={}, t={})".format(meth, b["exclusive"], b["overlap_thr"]))
    a.axvline(CHOSEN[v]["conf_thr"], ls="--", lw=0.8, color="k")
    a.set(xlabel="score threshold", ylabel="macro-F1", title="{}: VAL macro-F1 (dashed: chosen)".format(v), ylim=(0.5, 1.0)); a.grid(alpha=0.3); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(str(OUT / "val_threshold{}.png".format(SUF)), dpi=110); plt.show()

kinds = ["site", "climate", "day_type", "camera"]
fig, ax = plt.subplots(len(kinds), 2, figsize=(15, 3.1 * len(kinds)))
for row, k in zip(ax, kinds):
    gs = sorted(g for g in obs_groups if g.startswith(k + ":"))
    x = np.arange(len(gs)); w = 0.8 / max(1, len(RESULTS))
    for a, metric, lim in ((row[0], "accuracy", (0.8, 1.0)), (row[1], "false_vacant_rate", None)):
        for i, (v, r) in enumerate(RESULTS.items()):
            a.bar(x + i * w, [r["groups"][g][metric] for g in gs], w, label=v)
        a.set_xticks(x + w * (len(RESULTS) - 1) / 2)
        a.set_xticklabels(["{}\n(n={}){}".format(g.split(":", 1)[1].replace("CNRPark-EXT/", "CNR/"), len(img_count[g]),
                                                  " *" if RESULTS[next(iter(RESULTS))]["groups"][g]["note"] else "") for g in gs], fontsize=7)
        if lim: a.set_ylim(*lim)
        a.set_title("test slot {} by {} (* = small group)".format(metric, k), fontsize=10); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(str(OUT / "groups_test{}.png".format(SUF)), dpi=100); plt.show()

# overlay: one test image per site, smallest model, colours = outcome
from PIL import ImageDraw
from IPython.display import display
v0 = next(iter(MODELS)); c0 = CHOSEN[v0]
COL = {"TP": (220, 40, 40), "TN": (40, 200, 60), "FV": (255, 0, 255), "FO": (255, 150, 0), "unknown": (150, 150, 150)}
shown = set()
for k, n in enumerate(NAMES["test"]):
    site = ROW_OF[n]["site"]
    if site in shown or o["starts"][k] == o["starts"][k + 1]:
        continue
    shown.add(site)
    im = Image.open(str(LOCAL / "images" / "test" / n)).convert("RGB"); d = ImageDraw.Draw(im)
    D = DETS[(v0, "test")]
    for x0, y0, x1, y1, cf in D["dets"][D["starts"][k]:D["starts"][k + 1]]:
        if cf >= c0["conf_thr"]:
            d.rectangle([x0, y0, x1, y1], outline=(0, 220, 220), width=1)
    cnt = defaultdict(int)
    for i in range(o["starts"][k], o["starts"][k + 1]):
        gt_, pr_ = bool(o["gt"][i]), bool(PRED[(v0, "test")][i])
        kind = "unknown" if not o["valid"][i] else ("TP" if gt_ and pr_ else "TN" if not gt_ and not pr_ else "FV" if gt_ else "FO")
        cnt[kind] += 1
        d.polygon(geo.to_points(SLOTS[o["slot_id"][i]]["polygon"]), outline=COL[kind], width=3 if kind in ("FV", "FO") else 2)
    print("{} | {} | {} | magenta = false vacant, orange = false occupied, red/green = correct occupied/vacant, gray = unknown, cyan = detections".format(
        v0, n, dict(cnt)))
    display(im.resize((im.width * 2 // 3, im.height * 2 // 3)))

In [ ]:
# Per-slot output records (spec section 9 schema) for one test image; fixed model, so no controller mode or risk yet
k = next(i for i in range(len(NAMES["test"])) if o["starts"][i + 1] > o["starts"][i])
n = NAMES["test"][k]; r = ROW_OF[n]; c0 = CHOSEN[v0]
a, b = o["starts"][k], o["starts"][k + 1]
D = DETS[(v0, "test")]; dets = D["dets"][D["starts"][k]:D["starts"][k + 1]]
runtime = asc.assign_image(asc.prepare_slots([SLOTS[s]["polygon"] for s in o["slot_id"][a:b]]), dets,
                           {"method": c0["method"], "overlap_thr": c0["overlap_thr"], "exclusive": c0["exclusive"], "conf_thr": c0["conf_thr"]})
assert np.allclose([x["score"] for x in runtime], SCORES[(v0, "test")][a:b]), "runtime and batch association disagree"
EXAMPLE = []
for j, i in enumerate(range(a, b)):
    st = "unknown" if not o["valid"][i] else ("occupied" if runtime[j]["occupied"] else "vacant")
    sc = runtime[j]["score"]
    EXAMPLE.append(schema.slot_record(
        o["slot_id"][i], r["camera"], n, r["timestamp"], st, None if st == "unknown" else round(sc, 4),
        None if st == "unknown" else asc.uncertainty(sc, c0["conf_thr"]),
        [{"box_xyxy": [round(float(x), 1) for x in dets[d_][:4]], "conf": round(float(dets[d_][4]), 4)} for d_ in runtime[j]["support"]],
        None if st == "unknown" else n, None if st == "unknown" else r["timestamp"], None if st == "unknown" else 0,
        None if st == "unknown" else 0.0,
        schema.decision_reason(st, sc, c0["conf_thr"], c0["method"], len(runtime[j]["support"]), float(o["roi"][i]), MIN_ROI),
        controller_mode="fixed {} (no controller in this notebook)".format(v0)))
(OUT / "example_slot_outputs{}.json".format(SUF)).write_text(json.dumps(EXAMPLE, indent=1))
print(n, "|", len(EXAMPLE), "slots | first two records:")
print(json.dumps(EXAMPLE[:2], indent=1))

In [ ]:
# Finish: environment, status, clean up local copies in committed runs
from src.common.env_record import save_environment
save_environment(str(OUT / "environment.json"))
state = "COMPLETE" if not MISSING and len(RESULTS) == len(VARIANTS) else "PARTIAL"
write_status(STATUS, "07_slot_occupancy", RUN_ID, state, CFG_HASH,
             "smoke={} models={} missing={}".format(SMOKE, sorted(RESULTS), MISSING),
             extra={v: {"chosen_on_val": r["chosen_on_val"], "test_accuracy": r["test"]["accuracy"], "test_macro_f1": r["test"]["macro_f1"]}
                    for v, r in RESULTS.items()})
print(state, "| saved to", OUT, sorted(p.name for p in OUT.iterdir()))
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    shutil.rmtree(str(LOCAL), ignore_errors=True)
    print("removed staged copies")

## What to send back
Download the notebook after **Save & Run All** (SMOKE = False) and send it, or paste the output of: the settings cell, the slot registry cell, the detection cell, the VAL selection table, the TEST tables, and the overlays (say whether the slot outlines sit on the parking spaces and the colours look plausible).
`artifacts/occupancy/` (slot registry, cached detections, slot scores) is the input for notebooks 08-11, so **Save Version** with outputs.